# Qwen FG Seg debug (layout_gen_debug_outputs)

This notebook runs FG-seg for each id under `/home/ubuntu/layout_gen_debug_outputs`:
- `input_image.png`
- `layer_*_normalSize.png`

Outputs go to:
`REPO_ROOT/output/qwen_fg_seg_infer/debug/{id}`

Each image saves:
- model input image (RGBA)
- input RGB
- input alpha
- seg RGB
- seg alpha


In [1]:
import os
from pathlib import Path
import glob

import torch
from PIL import Image

from diffsynth.pipelines.qwen_image import QwenImagePipeline, ModelConfig


def _find_repo_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / "diffsynth").exists() and (parent / "pyproject.toml").exists():
            return parent
    return start


def _resolve_model_base(repo_root: Path) -> Path:
    env = os.environ.get("DIFFSYNTH_MODEL_BASE_PATH", "/mnt/local/diffsynth_models")
    if env:
        return Path(env)
    candidates = [
        repo_root / "diffsynth_models",
        Path("/mnt/lica-data-2/for_jjseol/diffsynth_models"),
        Path("/mnt/data/for_jjseol/diffsynth_models"),
    ]
    for cand in candidates:
        if (cand / "Qwen").exists():
            return cand
    return candidates[0]


REPO_ROOT = _find_repo_root(Path(".").resolve())
DEFAULT_MODEL_BASE = _resolve_model_base(REPO_ROOT)

LAYOUT_DEBUG_ROOT = Path("/home/ubuntu/layout_gen_debug_outputs")
OUTPUT_ROOT = REPO_ROOT / "output/qwen_fg_seg_infer/debug"

PROMPT = "Extract the foreground layer."
SEED = 123
STEPS = 10
USE_LORA = True
CUDA_DEVICE_INDEX = 1
EDIT_IMAGE_AUTO_RESIZE = True

# Optional: limit to specific ids (folder names). Empty list = all.
ID_ALLOWLIST = []

LORA_DIR = REPO_ROOT / "models/train/Qwen-Image-Edit-2511_lora_layered_vae_fg_seg"
LORA_PATH = LORA_DIR / "epoch-0.safetensors"

print(f"[repo] {REPO_ROOT}")
print(f"[model] base: {DEFAULT_MODEL_BASE}")
print(f"[inputs] {LAYOUT_DEBUG_ROOT}")
print(f"[outputs] {OUTPUT_ROOT}")


[repo] /home/ubuntu/DiffSynth-Studio
[model] base: /mnt/local/diffsynth_models
[inputs] /home/ubuntu/layout_gen_debug_outputs
[outputs] /home/ubuntu/DiffSynth-Studio/output/qwen_fg_seg_infer/debug


In [2]:
def _model_config_local_only(local_glob: str, label: str) -> ModelConfig:
    paths = sorted(glob.glob(local_glob))
    if not paths:
        raise FileNotFoundError(f"[model] Missing local weights for {label}: {local_glob}")
    path_value = paths if len(paths) > 1 else paths[0]
    print(f"[model] Using local weights: {path_value}")
    return ModelConfig(path=path_value)


def _aux_config_local_only(local_dir: Path, label: str) -> ModelConfig:
    if not local_dir.exists():
        raise FileNotFoundError(f"[aux] Missing local assets for {label}: {local_dir}")
    print(f"[aux] Using local assets: {local_dir}")
    return ModelConfig(path=str(local_dir))


if torch.cuda.is_available():
    torch.cuda.set_device(CUDA_DEVICE_INDEX)
    device = f"cuda:{CUDA_DEVICE_INDEX}"
    torch_dtype = torch.bfloat16
else:
    device = "cpu"
    torch_dtype = torch.float32

print(f"[device] {device}")

model_base = DEFAULT_MODEL_BASE
qwen_root = model_base / "Qwen"

pipe = QwenImagePipeline.from_pretrained(
    torch_dtype=torch_dtype,
    device=device,
    model_configs=[
        _model_config_local_only(
            str(qwen_root / "Qwen-Image-Edit-2511" / "transformer" / "diffusion_pytorch_model-*.safetensors"),
            "Qwen-Image-Edit-2511 transformer",
        ),
        _model_config_local_only(
            str(qwen_root / "Qwen-Image" / "text_encoder" / "model*.safetensors"),
            "Qwen-Image text encoder",
        ),
        _model_config_local_only(
            str(qwen_root / "Qwen-Image-Layered" / "vae" / "diffusion_pytorch_model.safetensors"),
            "Qwen-Image-Layered VAE",
        ),
    ],
    tokenizer_config=None,
    processor_config=_aux_config_local_only(qwen_root / "Qwen-Image-Edit" / "processor", "Qwen-Image-Edit processor"),
)

if USE_LORA:
    if LORA_DIR.is_dir():
        lora_candidates = sorted(LORA_DIR.glob("*.safetensors"), key=lambda p: p.stat().st_mtime)
        if not lora_candidates:
            raise FileNotFoundError(f"No LoRA checkpoints found in: {LORA_DIR}")
        LORA_PATH = lora_candidates[-1]

    if not LORA_PATH.exists():
        raise FileNotFoundError(f"LoRA checkpoint not found: {LORA_PATH}")

    print(f"[lora] Using LoRA: {LORA_PATH}")
    pipe.load_lora(pipe.dit, str(LORA_PATH))


[device] cuda:1
[model] Using local weights: ['/mnt/local/diffsynth_models/Qwen/Qwen-Image-Edit-2511/transformer/diffusion_pytorch_model-00001-of-00005.safetensors', '/mnt/local/diffsynth_models/Qwen/Qwen-Image-Edit-2511/transformer/diffusion_pytorch_model-00002-of-00005.safetensors', '/mnt/local/diffsynth_models/Qwen/Qwen-Image-Edit-2511/transformer/diffusion_pytorch_model-00003-of-00005.safetensors', '/mnt/local/diffsynth_models/Qwen/Qwen-Image-Edit-2511/transformer/diffusion_pytorch_model-00004-of-00005.safetensors', '/mnt/local/diffsynth_models/Qwen/Qwen-Image-Edit-2511/transformer/diffusion_pytorch_model-00005-of-00005.safetensors']
[model] Using local weights: ['/mnt/local/diffsynth_models/Qwen/Qwen-Image/text_encoder/model-00001-of-00004.safetensors', '/mnt/local/diffsynth_models/Qwen/Qwen-Image/text_encoder/model-00002-of-00004.safetensors', '/mnt/local/diffsynth_models/Qwen/Qwen-Image/text_encoder/model-00003-of-00004.safetensors', '/mnt/local/diffsynth_models/Qwen/Qwen-Image/

Loaded model: {
    "model_name": "qwen_image_dit",
    "model_class": "diffsynth.models.qwen_image_dit.QwenImageDiT",
    "extra_kwargs": null
}
Loading models from: [
    "/mnt/local/diffsynth_models/Qwen/Qwen-Image/text_encoder/model-00001-of-00004.safetensors",
    "/mnt/local/diffsynth_models/Qwen/Qwen-Image/text_encoder/model-00002-of-00004.safetensors",
    "/mnt/local/diffsynth_models/Qwen/Qwen-Image/text_encoder/model-00003-of-00004.safetensors",
    "/mnt/local/diffsynth_models/Qwen/Qwen-Image/text_encoder/model-00004-of-00004.safetensors"
]
Loaded model: {
    "model_name": "qwen_image_text_encoder",
    "model_class": "diffsynth.models.qwen_image_text_encoder.QwenImageTextEncoder",
    "extra_kwargs": null
}
Loading models from: "/mnt/local/diffsynth_models/Qwen/Qwen-Image-Layered/vae/diffusion_pytorch_model.safetensors"
Loaded model: {
    "model_name": "qwen_image_vae",
    "model_class": "diffsynth.models.qwen_image_vae.QwenImageVAE",
    "extra_kwargs": {
        "image

In [3]:
def _split_channels(image: Image.Image):
    rgb = image.convert("RGB")
    if "A" in image.getbands():
        alpha = image.getchannel("A")
    else:
        alpha = Image.new("L", image.size, 255)
    return rgb, alpha


def _checkerboard_rgba(size, tile=32):
    w, h = size
    light = (200, 200, 200, 255)
    dark = (160, 160, 160, 255)
    bg = Image.new("RGBA", size, light)
    tile_img = Image.new("RGBA", (tile, tile), dark)
    for y in range(0, h, tile):
        for x in range(0, w, tile):
            if (x // tile + y // tile) % 2 == 1:
                bg.paste(tile_img, (x, y))
    return bg


def _iter_id_dirs(root: Path):
    if not root.exists():
        raise FileNotFoundError(f"Layout debug root not found: {root}")
    for path in sorted(root.iterdir()):
        if path.is_dir():
            yield path


def run_fg_seg_on_image(image_path: Path, out_dir: Path, seed: int) -> dict:
    edit_image = Image.open(image_path).convert("RGBA")
    width, height = edit_image.size

    result = pipe(
        PROMPT,
        edit_image=edit_image,
        seed=seed,
        num_inference_steps=STEPS,
        height=height,
        width=width,
        edit_image_auto_resize=EDIT_IMAGE_AUTO_RESIZE,
        zero_cond_t=True,
    )

    output_rgba = result.convert("RGBA")
    if edit_image.size != output_rgba.size:
        edit_image = edit_image.resize(output_rgba.size, Image.LANCZOS)

    out_dir.mkdir(parents=True, exist_ok=True)
    stem = image_path.stem

    input_rgb, input_alpha = _split_channels(edit_image)
    output_rgb, output_alpha = _split_channels(output_rgba)

    # Canvas: model input (RGBA on checker) | input RGB | input alpha | seg RGB | seg alpha
    tile_w, tile_h = edit_image.size
    canvas = Image.new("RGB", (tile_w * 5, tile_h), (0, 0, 0))
    checker = _checkerboard_rgba(edit_image.size)
    edit_on_checker = Image.alpha_composite(checker, edit_image)
    tiles = [
        edit_on_checker.convert("RGB"),
        input_rgb,
        input_alpha.convert("RGB"),
        output_rgb,
        output_alpha.convert("RGB"),
    ]
    for idx, tile in enumerate(tiles):
        canvas.paste(tile, (tile_w * idx, 0))

    canvas_path = out_dir / f"{stem}_canvas.png"
    canvas.save(canvas_path)

    return {
        "canvas": canvas_path,
    }


if ID_ALLOWLIST:
    id_dirs = [LAYOUT_DEBUG_ROOT / id_name for id_name in ID_ALLOWLIST]
else:
    id_dirs = list(_iter_id_dirs(LAYOUT_DEBUG_ROOT))

if not id_dirs:
    raise FileNotFoundError(f"No id directories found under: {LAYOUT_DEBUG_ROOT}")

for id_dir in id_dirs:
    if not id_dir.exists():
        print(f"[warn] id dir missing: {id_dir}")
        continue

    out_dir = OUTPUT_ROOT / id_dir.name
    print(f"[id] {id_dir.name} -> {out_dir}")

    image_paths = []
    input_path = id_dir / "input_image.png"
    if input_path.exists():
        image_paths.append(input_path)
    else:
        print(f"[warn] missing input_image.png: {id_dir}")

    layer_paths = sorted(id_dir.glob("layer_*_normalSize.png"))
    if not layer_paths:
        print(f"[warn] no layer_*_normalSize.png: {id_dir}")
    image_paths.extend(layer_paths)

    if not image_paths:
        print(f"[skip] no images: {id_dir}")
        continue

    for idx, image_path in enumerate(image_paths):
        outputs = run_fg_seg_on_image(image_path, out_dir, seed=SEED + idx)
        print(f"[done] {image_path.name} -> {outputs['canvas'].name}")


[id] 38204799-72bb-419a-8652-12a0c1388c76_debug_20260212_161325 -> /home/ubuntu/DiffSynth-Studio/output/qwen_fg_seg_infer/debug/38204799-72bb-419a-8652-12a0c1388c76_debug_20260212_161325


100%|██████████| 10/10 [00:24<00:00,  2.44s/it]


[done] input_image.png -> input_image_canvas.png


100%|██████████| 10/10 [00:17<00:00,  1.73s/it]


[done] layer_1_normalSize.png -> layer_1_normalSize_canvas.png


100%|██████████| 10/10 [00:17<00:00,  1.73s/it]


[done] layer_2_normalSize.png -> layer_2_normalSize_canvas.png


 30%|███       | 3/10 [00:05<00:13,  1.90s/it]


KeyboardInterrupt: 

# debugg-trick

In [4]:
# debug_trick: set RGB to black where alpha < 10 before model input
DEBUG_TRICK_OUTPUT_ROOT = REPO_ROOT / "output/qwen_fg_seg_infer/debug_trick"


def _apply_alpha_zero_black(rgba: Image.Image) -> Image.Image:
    rgba = rgba.convert("RGBA")
    alpha = rgba.getchannel("A")
    rgb = rgba.convert("RGB")
    zero_mask = alpha.point(lambda a: 255 if a < 10 else 0)
    black = Image.new("RGB", rgba.size, (0, 0, 0))
    rgb_black = Image.composite(black, rgb, zero_mask)
    r, g, b = rgb_black.split()
    return Image.merge("RGBA", (r, g, b, alpha))


def run_fg_seg_on_image_debug_trick(image_path: Path, out_dir: Path, seed: int) -> dict:
    edit_image = Image.open(image_path).convert("RGBA")
    edit_image_trick = _apply_alpha_zero_black(edit_image)
    width, height = edit_image_trick.size

    result = pipe(
        PROMPT,
        edit_image=edit_image_trick,
        seed=seed,
        num_inference_steps=STEPS,
        height=height,
        width=width,
        edit_image_auto_resize=EDIT_IMAGE_AUTO_RESIZE,
        zero_cond_t=True,
    )

    output_rgba = result.convert("RGBA")
    if edit_image_trick.size != output_rgba.size:
        edit_image_trick = edit_image_trick.resize(output_rgba.size, Image.LANCZOS)

    out_dir.mkdir(parents=True, exist_ok=True)
    stem = image_path.stem

    input_rgb, input_alpha = _split_channels(edit_image_trick)
    output_rgb, output_alpha = _split_channels(output_rgba)

    # Canvas: model input (RGBA on checker) | input RGB | input alpha | seg RGB | seg alpha
    tile_w, tile_h = edit_image_trick.size
    canvas = Image.new("RGB", (tile_w * 5, tile_h), (0, 0, 0))
    checker = _checkerboard_rgba(edit_image_trick.size)
    edit_on_checker = Image.alpha_composite(checker, edit_image_trick)
    tiles = [
        edit_on_checker.convert("RGB"),
        input_rgb,
        input_alpha.convert("RGB"),
        output_rgb,
        output_alpha.convert("RGB"),
    ]
    for idx, tile in enumerate(tiles):
        canvas.paste(tile, (tile_w * idx, 0))

    canvas_path = out_dir / f"{stem}_canvas.png"
    canvas.save(canvas_path)

    return {
        "canvas": canvas_path,
    }


if ID_ALLOWLIST:
    id_dirs = [LAYOUT_DEBUG_ROOT / id_name for id_name in ID_ALLOWLIST]
else:
    id_dirs = list(_iter_id_dirs(LAYOUT_DEBUG_ROOT))

if not id_dirs:
    raise FileNotFoundError(f"No id directories found under: {LAYOUT_DEBUG_ROOT}")

for id_dir in id_dirs:
    if not id_dir.exists():
        print(f"[warn] id dir missing: {id_dir}")
        continue

    out_dir = DEBUG_TRICK_OUTPUT_ROOT / id_dir.name
    print(f"[debug_trick] {id_dir.name} -> {out_dir}")

    image_paths = []
    input_path = id_dir / "input_image.png"
    if input_path.exists():
        image_paths.append(input_path)
    else:
        print(f"[warn] missing input_image.png: {id_dir}")

    layer_paths = sorted(id_dir.glob("layer_*_normalSize.png"))
    if not layer_paths:
        print(f"[warn] no layer_*_normalSize.png: {id_dir}")
    image_paths.extend(layer_paths)

    if not image_paths:
        print(f"[skip] no images: {id_dir}")
        continue

    for idx, image_path in enumerate(image_paths):
        outputs = run_fg_seg_on_image_debug_trick(image_path, out_dir, seed=SEED + idx)
        print(f"[done] {image_path.name} -> {outputs['canvas'].name}")


[debug_trick] 38204799-72bb-419a-8652-12a0c1388c76_debug_20260212_161325 -> /home/ubuntu/DiffSynth-Studio/output/qwen_fg_seg_infer/debug_trick/38204799-72bb-419a-8652-12a0c1388c76_debug_20260212_161325


100%|██████████| 10/10 [00:24<00:00,  2.48s/it]


[done] input_image.png -> input_image_canvas.png


100%|██████████| 10/10 [00:17<00:00,  1.74s/it]


[done] layer_1_normalSize.png -> layer_1_normalSize_canvas.png


...

[done] layer_2_normalSize.png -> layer_2_normalSize_canvas.png


100%|██████████| 10/10 [00:17<00:00,  1.74s/it]


[done] layer_3_normalSize.png -> layer_3_normalSize_canvas.png


100%|██████████| 10/10 [00:17<00:00,  1.74s/it]


[done] layer_4_normalSize.png -> layer_4_normalSize_canvas.png
